# Texture-choice overview

This notebook presents the current LocustVR texture-choice experiment in four complementary figures: a trial overview, pairwise preference indices, choice composition, and texture-aligned trajectories. Bait texture is retained in the exported table but is not used for grouping or plotting.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

plt.style.use('default')

save_path = Path('/Users/jonny/Desktop/Uni/Master/collevtive behavior/VR_analysis/collor_choice')
data_file = save_path / 'locustvr_data.pkl'
save_figures = True

TEXTURE_ORDER = ['black', 'yellow', 'gregarious']
TEXTURE_COLORS = {
    'black': '#252525',
    'yellow': '#D9A514',
    'gregarious': '#3579A8',
    'neutral': '#E5E7EB',
}
MIXED_PAIR_ORDER = ['black–yellow', 'black–gregarious', 'yellow–gregarious']
IDENTICAL_PAIR_ORDER = ['black–black', 'yellow–yellow', 'gregarious–gregarious']
ALL_PAIR_ORDER = MIXED_PAIR_ORDER + IDENTICAL_PAIR_ORDER

if not data_file.is_file():
    raise FileNotFoundError(
        f'{data_file} does not exist. Run locustVR_data_preparation.ipynb first.'
    )
data = pd.read_pickle(data_file)
required = {
    'ts', 'trial_id', 'state_type', 'animal_id', 'heading_rel', 'trial_label',
    'condition_code', 'label_state', 'left_texture', 'right_texture', 'bait_texture'
}
missing = sorted(required.difference(data.columns))
if missing:
    raise KeyError('Missing prepared-data columns: ' + ', '.join(missing))
unexpected = (
    set(data['left_texture'].dropna())
    | set(data['right_texture'].dropna())
    | set(data['bait_texture'].dropna())
) - set(TEXTURE_ORDER)
if unexpected:
    raise ValueError(f'Unexpected texture names: {sorted(unexpected)}')
save_path.mkdir(parents=True, exist_ok=True)
print(f'Loaded {len(data):,} rows from {data_file}')

## Decision rule

The decision is based on the mean relative heading during the 20-second test phase. In the experiment coordinate convention, −75° to −45° selects `left_texture`, +45° to +75° selects `right_texture`, and all other means are neutral. Set `FLIP_SIDES = True` to combine side-swapped trials by texture in Figure 4. Set it to `False` for a texture-independent control of physical left-versus-right side bias.

In [ ]:
def canonical_pair(left_texture, right_texture):
    rank = {texture: index for index, texture in enumerate(TEXTURE_ORDER)}
    first, second = sorted([left_texture, right_texture], key=rank.get)
    return f'{first}–{second}'


def classify_side(angle, lower_threshold=45, upper_threshold=75):
    if np.radians(-upper_threshold) < angle < np.radians(-lower_threshold):
        return 'left'
    if np.radians(lower_threshold) < angle < np.radians(upper_threshold):
        return 'right'
    return 'neutral'


def classify_trial_choices(
    df, time_window=(0, 20), lower_threshold=45, upper_threshold=75
):
    if not 0 <= lower_threshold < upper_threshold <= 180:
        raise ValueError('Thresholds must satisfy 0 <= lower < upper <= 180.')
    start_frame, end_frame = (int(value * 100) for value in time_window)
    choice_phase = df.loc[
        df['ts'].between(start_frame, end_frame) & df['state_type'].isin([1, 2])
    ].copy()
    if choice_phase.empty:
        raise ValueError('No test-phase rows were found in the requested window.')

    metadata = [
        'trial_label', 'condition_code', 'label_state',
        'left_texture', 'right_texture', 'bait_texture'
    ]
    choices = (
        choice_phase.groupby(['animal_id', 'trial_id'], as_index=False)
        .agg(
            mean_heading=('heading_rel', 'mean'),
            n_frames=('heading_rel', 'count'),
            **{column: (column, 'first') for column in metadata},
        )
    )
    choices['choice_side'] = choices['mean_heading'].apply(
        classify_side, args=(lower_threshold, upper_threshold)
    )
    choices['chosen_texture'] = np.select(
        [choices['choice_side'].eq('left'), choices['choice_side'].eq('right')],
        [choices['left_texture'], choices['right_texture']],
        default='neutral',
    )
    choices['texture_pair'] = [
        canonical_pair(left, right)
        for left, right in zip(choices['left_texture'], choices['right_texture'])
    ]
    choices['identical_pair'] = choices['left_texture'].eq(choices['right_texture'])
    return choices.sort_values(['animal_id', 'trial_id']).reset_index(drop=True)


def preference_index(values):
    non_neutral = pd.Series(values).dropna()
    non_neutral = non_neutral[non_neutral != 0]
    return np.nan if non_neutral.empty else non_neutral.mean()


def pairwise_preference_indices(choices):
    mixed = choices.loc[~choices['identical_pair']].copy()
    pair_parts = mixed['texture_pair'].str.split('–', expand=True)
    mixed['first_texture'] = pair_parts[0]
    mixed['second_texture'] = pair_parts[1]
    mixed['pair_choice'] = np.select(
        [
            mixed['chosen_texture'].eq(mixed['first_texture']),
            mixed['chosen_texture'].eq(mixed['second_texture']),
        ],
        [-1, 1], default=0,
    )
    return (
        mixed.groupby(['animal_id', 'texture_pair'])['pair_choice']
        .apply(preference_index).rename('preference_index').reset_index()
    )


def texture_aligned_trial_bins(
    df, time_window=(0, 20), bin_size_s=0.5, flip_sides=True
):
    start_frame, end_frame = (int(value * 100) for value in time_window)
    plot_data = df.loc[
        df['ts'].between(start_frame, end_frame)
        & df['state_type'].isin([1, 2])
    ].copy()
    if flip_sides:
        rank = {texture: index for index, texture in enumerate(TEXTURE_ORDER)}
        left_rank = plot_data['left_texture'].map(rank)
        right_rank = plot_data['right_texture'].map(rank)
        left_is_first = left_rank < right_rank
        plot_data['first_texture'] = np.where(
            left_is_first, plot_data['left_texture'], plot_data['right_texture']
        )
        plot_data['second_texture'] = np.where(
            left_is_first, plot_data['right_texture'], plot_data['left_texture']
        )
        plot_data['texture_pair'] = (
            plot_data['first_texture'] + '–' + plot_data['second_texture']
        )
        flip_factor = np.where(
            plot_data['left_texture'].eq(plot_data['first_texture']), 1.0, -1.0
        )
    else:
        plot_data['first_texture'] = plot_data['left_texture']
        plot_data['second_texture'] = plot_data['right_texture']
        plot_data['texture_pair'] = (
            plot_data['left_texture'] + '→' + plot_data['right_texture']
        )
        flip_factor = 1.0
    plot_data['aligned_heading_deg'] = np.degrees(plot_data['heading_rel']) * flip_factor
    plot_data['time_s'] = plot_data['ts'] / 100
    plot_data['time_bin'] = (
        (plot_data['time_s'] // bin_size_s) * bin_size_s + bin_size_s / 2
    )
    return (
        plot_data.groupby([
            'animal_id', 'trial_id', 'texture_pair',
            'first_texture', 'second_texture', 'time_bin'
        ], as_index=False)['aligned_heading_deg'].mean()
    )

In [ ]:
TIME_WINDOW = (0, 20)
LOWER_THRESHOLD = 45
UPPER_THRESHOLD = 75

df_choice = classify_trial_choices(
    data, TIME_WINDOW, LOWER_THRESHOLD, UPPER_THRESHOLD
)
pairwise_pi = pairwise_preference_indices(df_choice)

overview = pd.Series({
    'animals': df_choice['animal_id'].nunique(),
    'trials': len(df_choice),
    'mixed-pair trials': int((~df_choice['identical_pair']).sum()),
    'neutral trials': int(df_choice['chosen_texture'].eq('neutral').sum()),
    'neutral percentage': round(100 * df_choice['chosen_texture'].eq('neutral').mean(), 1),
}, name='value')
display(overview.to_frame())

## Figure 1 — Trial-level overview

One cell represents one completed trial. White cells are missing trial numbers; grey cells are neutral decisions.

In [ ]:
def plot_trial_choice_matrix(choices, save_fig=False):
    category_order = TEXTURE_ORDER + ['neutral']
    category_code = {name: index for index, name in enumerate(category_order)}
    matrix = (
        choices.assign(choice_code=choices['chosen_texture'].map(category_code))
        .pivot(index='animal_id', columns='trial_id', values='choice_code')
        .sort_index()
    )
    values = np.ma.masked_invalid(matrix.to_numpy(dtype=float))
    cmap = ListedColormap([TEXTURE_COLORS[name] for name in category_order])
    cmap.set_bad('#FFFFFF')

    fig_height = max(4.5, 0.28 * len(matrix.index) + 1.8)
    fig, ax = plt.subplots(figsize=(13, fig_height))
    ax.imshow(values, aspect='auto', interpolation='none', cmap=cmap, vmin=-0.5, vmax=3.5)
    x_step = max(1, int(np.ceil(len(matrix.columns) / 12)))
    x_positions = np.arange(0, len(matrix.columns), x_step)
    ax.set_xticks(x_positions, matrix.columns[x_positions])
    ax.set_yticks(np.arange(len(matrix.index)), matrix.index)
    ax.set_xlabel('Trial ID')
    ax.set_ylabel('Animal ID')
    ax.set_title('Texture selected in every trial')
    handles = [
        Patch(facecolor=TEXTURE_COLORS[name], label=name) for name in category_order
    ] + [Patch(facecolor='white', edgecolor='#999999', label='missing')]
    ax.legend(handles=handles, ncol=5, loc='upper center', bbox_to_anchor=(0.5, 1.16), frameon=False)
    fig.tight_layout()
    if save_fig:
        fig.savefig(save_path / 'overview_trial_choices.pdf', bbox_inches='tight')
    plt.show()


plot_trial_choice_matrix(df_choice, save_fig=save_figures)

## Figure 2 — Pairwise preference by animal

Each grey point is one animal. Diamonds show the mean; vertical lines show a deterministic bootstrap 95% confidence interval. Negative values favour the first texture named below the panel, positive values favour the second. Neutral trials are excluded from the denominator.

In [ ]:
def bootstrap_mean_ci(values, n_boot=5000, seed=42):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    low, high = np.quantile(samples, [0.025, 0.975])
    return values.mean(), low, high


def plot_pairwise_preference(pi, save_fig=False):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), sharey=True)
    rng = np.random.default_rng(7)
    for ax, pair in zip(axes, MIXED_PAIR_ORDER):
        first_texture, second_texture = pair.split('–')
        values = pi.loc[pi['texture_pair'] == pair, 'preference_index'].dropna().to_numpy()
        if len(values) >= 2 and np.ptp(values) > 0:
            violin = ax.violinplot(values, positions=[0], widths=0.8, showextrema=False)
            for body in violin['bodies']:
                body.set_facecolor('#D9DDE2')
                body.set_edgecolor('none')
                body.set_alpha(0.65)
        jitter = rng.uniform(-0.18, 0.18, len(values))
        ax.scatter(jitter, values, color='#5F6874', alpha=0.7, s=30, zorder=3)
        mean, low, high = bootstrap_mean_ci(values)
        if np.isfinite(mean):
            ax.errorbar(
                0, mean, yerr=[[mean - low], [high - mean]],
                fmt='D', color='black', markersize=6, capsize=4, linewidth=1.6, zorder=4,
            )
        ax.axhline(0, color='#777777', linestyle='--', linewidth=0.9)
        ax.text(-0.44, -0.94, f'− {first_texture}', color=TEXTURE_COLORS[first_texture], va='center')
        ax.text(-0.44, 0.94, f'+ {second_texture}', color=TEXTURE_COLORS[second_texture], va='center')
        ax.set_title(pair.replace('–', ' ↔ '))
        ax.set_xlim(-0.5, 0.5)
        ax.set_ylim(-1.08, 1.08)
        ax.set_xticks([])
        ax.text(0.44, -0.94, f'n = {len(values)}', ha='right', va='center', color='#555555')
        ax.grid(axis='y', alpha=0.18)
    axes[0].set_ylabel('Pairwise preference index')
    fig.suptitle('Individual preference and group estimate', y=1.02)
    fig.tight_layout()
    if save_fig:
        fig.savefig(save_path / 'pairwise_preference_overview.pdf', bbox_inches='tight')
    plt.show()


plot_pairwise_preference(pairwise_pi, save_fig=save_figures)

## Figure 3 — Choice composition

Bars show texture shares among classified decisions. Neutral trials are not part of the bars; their mean per-animal percentage is shown only as a small annotation. Mixed pairs are shown first, followed by the identical-texture controls.

In [ ]:
def animal_choice_proportions(choices):
    categories = TEXTURE_ORDER + ['neutral']
    counts = (
        choices.groupby(['animal_id', 'texture_pair', 'chosen_texture']).size()
        .unstack('chosen_texture', fill_value=0)
        .reindex(columns=categories, fill_value=0)
    )
    totals = counts.sum(axis=1)
    decided = counts[TEXTURE_ORDER].sum(axis=1).replace(0, np.nan)
    result = counts[TEXTURE_ORDER].div(decided, axis=0)
    result['neutral_rate'] = counts['neutral'].div(totals)
    return result.reset_index()


def plot_choice_composition(choices, save_fig=False):
    categories = TEXTURE_ORDER
    per_animal = animal_choice_proportions(choices)
    present_pairs = [pair for pair in ALL_PAIR_ORDER if pair in set(per_animal['texture_pair'])]
    grouped = per_animal.groupby('texture_pair')
    means = grouped[categories].mean().reindex(present_pairs)
    neutral_rates = grouped['neutral_rate'].mean().reindex(present_pairs)
    fig, ax = plt.subplots(figsize=(10, 5.8))
    y = np.arange(len(means))
    left = np.zeros(len(means))
    for texture in categories:
        values = means[texture].to_numpy()
        ax.barh(y, values, left=left, color=TEXTURE_COLORS[texture], label=texture)
        for row, (start, value) in enumerate(zip(left, values)):
            if value >= 0.08:
                text_color = 'white' if texture in ('black', 'gregarious') else '#222222'
                ax.text(start + value / 2, row, f'{100 * value:.0f}%', ha='center', va='center', color=text_color)
        left += values
    ax.set_yticks(y, [pair.replace('–', ' ↔ ') for pair in means.index])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    for row, neutral_rate in enumerate(neutral_rates):
        ax.text(1.015, row, f'neutral {neutral_rate:.0%}', transform=ax.get_yaxis_transform(), ha='left', va='center', color='#7A7A7A', fontsize=9, clip_on=False)
    ax.set_xlabel('Mean share among classified decisions per animal')
    ax.set_title('Texture choices among classified decisions')
    ax.axhline(2.5, color='#999999', linewidth=0.8)
    ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.14), frameon=False)
    fig.tight_layout(rect=(0, 0, 0.9, 1))
    if save_fig:
        fig.savefig(save_path / 'choice_composition.pdf', bbox_inches='tight')
    plt.show()
    return per_animal


animal_choice_summary = plot_choice_composition(df_choice, save_fig=save_figures)

## Figure 4 — Texture-aligned trajectories

Every thin grey line is one trial. The black line is the trial mean and the grey band is its approximate 95% confidence interval (`mean ± 1.96 × SEM`). With `FLIP_SIDES = True`, side-swapped trials are combined into six texture panels. With `False`, all trials retain their physical orientation and are summarized as a left-versus-right side-bias control; neutral decisions are excluded from the side-preference index.

In [ ]:
def plot_texture_aligned_trajectories(
    bins, lower_threshold=45, upper_threshold=75, time_window=(0, 20),
    flip_sides=True, save_fig=False
):
    if flip_sides:
        pair_order = ALL_PAIR_ORDER
        n_rows, figure_height = 2, 9
    else:
        pair_order = [
            f'{left}→{right}'
            for left in TEXTURE_ORDER for right in TEXTURE_ORDER
        ]
        n_rows, figure_height = 3, 12
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, figure_height), sharex=True, sharey=True)
    decision_angle = (lower_threshold + upper_threshold) / 2
    for ax, pair in zip(axes.flat, pair_order):
        separator = '–' if flip_sides else '→'
        first_texture, second_texture = pair.split(separator)
        subset = bins.loc[bins['texture_pair'] == pair]
        for (_, _), trial in subset.groupby(['animal_id', 'trial_id']):
            trial = trial.sort_values('time_bin')
            ax.plot(
                trial['time_bin'], trial['aligned_heading_deg'],
                color='#6B7280', alpha=0.08, linewidth=0.65,
            )
        stats = subset.groupby('time_bin')['aligned_heading_deg'].agg(['mean', 'sem', 'count'])
        if not stats.empty:
            x = stats.index.to_numpy(dtype=float)
            mean = stats['mean'].to_numpy(dtype=float)
            ci = 1.96 * stats['sem'].fillna(0).to_numpy(dtype=float)
            ax.fill_between(x, mean - ci, mean + ci, color='#555555', alpha=0.2, linewidth=0)
            ax.plot(x, mean, color='black', linewidth=2.2, zorder=4)
        ax.axhline(-decision_angle, color=TEXTURE_COLORS[first_texture], linestyle=':', linewidth=1.0, alpha=0.65)
        ax.axhline(decision_angle, color=TEXTURE_COLORS[second_texture], linestyle=':', linewidth=1.0, alpha=0.65)
        ax.axhline(0, color='#A0A0A0', linewidth=0.6)
        negative_label = f'− {first_texture}' if flip_sides else f'← {first_texture}'
        positive_label = f'+ {second_texture}' if flip_sides else f'{second_texture} →'
        ax.text(0.4, -decision_angle, negative_label, color=TEXTURE_COLORS[first_texture], va='bottom', fontweight='bold')
        ax.text(0.4, decision_angle, positive_label, color=TEXTURE_COLORS[second_texture], va='bottom', fontweight='bold')
        ax.text(time_window[1] - 0.4, 164, f'n = {subset[["animal_id", "trial_id"]].drop_duplicates().shape[0]} trials', ha='right', va='top', color='#555555')
        if flip_sides:
            title = pair.replace('–', ' ↔ ') if first_texture != second_texture else f'{first_texture} on both sides'
        else:
            title = f'L: {first_texture} | R: {second_texture}'
        ax.set_title(title)
        ax.set_xlim(time_window)
        ax.set_ylim(-180, 180)
        ax.grid(alpha=0.14)
    y_label = 'Texture-aligned heading (degrees)' if flip_sides else 'Heading relative to trial start (degrees)'
    fig.supylabel(y_label)
    fig.supxlabel('Time in test phase (s)')
    title_mode = 'side-flipped by texture' if flip_sides else 'original left/right orientation'
    fig.suptitle(f'Individual trajectories and group time course — {title_mode}', y=0.995)
    handles = [
        Line2D([0], [0], color='#6B7280', alpha=0.5, linewidth=1, label='individual trial'),
        Line2D([0], [0], color='black', linewidth=2.2, label='mean'),
        Patch(facecolor='#777777', alpha=0.2, label='approx. 95% CI'),
    ]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.965), ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.91))
    if save_fig:
        file_name = (
            'texture_aligned_trajectories.pdf'
            if flip_sides else 'left_right_trajectories.pdf'
        )
        fig.savefig(save_path / file_name, bbox_inches='tight')
    plt.show()


def plot_side_bias_control(
    bins, choices, lower_threshold=45, upper_threshold=75,
    time_window=(0, 20), save_fig=False
):
    left_color, right_color = '#7A5195', '#D66B3D'
    decision_angle = (lower_threshold + upper_threshold) / 2
    fig, (ax_time, ax_bias) = plt.subplots(
        1, 2, figsize=(13, 5.2), gridspec_kw={'width_ratios': [2.2, 1]}
    )

    for (_, _), trial in bins.groupby(['animal_id', 'trial_id']):
        trial = trial.sort_values('time_bin')
        ax_time.plot(
            trial['time_bin'], trial['aligned_heading_deg'],
            color='#6B7280', alpha=0.06, linewidth=0.65,
        )
    stats = bins.groupby('time_bin')['aligned_heading_deg'].agg(['mean', 'sem'])
    x = stats.index.to_numpy(dtype=float)
    mean = stats['mean'].to_numpy(dtype=float)
    ci = 1.96 * stats['sem'].fillna(0).to_numpy(dtype=float)
    ax_time.fill_between(
        x, mean - ci, mean + ci, color='#555555', alpha=0.2, linewidth=0
    )
    ax_time.plot(x, mean, color='black', linewidth=2.2, zorder=4)
    ax_time.axhline(-decision_angle, color=left_color, linestyle=':', linewidth=1.1)
    ax_time.axhline(decision_angle, color=right_color, linestyle=':', linewidth=1.1)
    ax_time.axhline(0, color='#A0A0A0', linewidth=0.6)
    ax_time.text(0.4, -decision_angle, '← LEFT', color=left_color, va='bottom', fontweight='bold')
    ax_time.text(0.4, decision_angle, 'RIGHT →', color=right_color, va='bottom', fontweight='bold')
    n_trials = bins[['animal_id', 'trial_id']].drop_duplicates().shape[0]
    ax_time.text(
        time_window[1] - 0.4, 164, f'n = {n_trials} trials',
        ha='right', va='top', color='#555555'
    )
    ax_time.set(
        xlim=time_window, ylim=(-180, 180),
        xlabel='Time in test phase (s)',
        ylabel='Heading relative to trial start (degrees)',
        title='Original orientation across all trials',
    )
    ax_time.grid(alpha=0.14)

    side_values = choices['choice_side'].map({'left': -1.0, 'right': 1.0})
    side_pi = (
        choices.assign(side_value=side_values)
        .groupby('animal_id')['side_value'].mean()
        .dropna()
    )
    values = side_pi.to_numpy(dtype=float)
    rng = np.random.default_rng(42)
    jitter = rng.uniform(-0.12, 0.12, len(values))
    point_colors = np.where(values < 0, left_color, right_color)
    ax_bias.scatter(jitter, values, c=point_colors, alpha=0.75, s=38, zorder=3)
    if len(values):
        samples = rng.choice(
            values, size=(5000, len(values)), replace=True
        ).mean(axis=1)
        group_mean = values.mean()
        low, high = np.quantile(samples, [0.025, 0.975])
        ax_bias.errorbar(
            0, group_mean, yerr=[[group_mean - low], [high - group_mean]],
            fmt='D', color='black', markersize=6, capsize=4,
            linewidth=1.6, zorder=4,
        )
    ax_bias.axhline(0, color='#777777', linestyle='--', linewidth=0.9)
    ax_bias.text(-0.28, -0.95, 'LEFT', color=left_color, va='center', fontweight='bold')
    ax_bias.text(-0.28, 0.95, 'RIGHT', color=right_color, va='center', fontweight='bold')
    ax_bias.text(
        0.28, -0.95, f'n = {len(values)} animals',
        ha='right', va='center', color='#555555'
    )
    ax_bias.set(
        xlim=(-0.3, 0.3), ylim=(-1.08, 1.08),
        ylabel='Side preference index', title='Side preference by animal',
    )
    ax_bias.set_xticks([])
    ax_bias.grid(axis='y', alpha=0.14)

    fig.suptitle('Side-bias control — original left/right orientation', y=1.02)
    fig.tight_layout()
    if save_fig:
        fig.savefig(save_path / 'side_bias_control.pdf', bbox_inches='tight')
    plt.show()
    return side_pi.rename('side_preference_index').reset_index()


# True = texture-aligned plots; False = left/right side-bias control.
# Change only this value, then run this cell again.
FLIP_SIDES = True

trial_bins = texture_aligned_trial_bins(
    data, TIME_WINDOW, bin_size_s=0.5, flip_sides=FLIP_SIDES
)
if FLIP_SIDES:
    plot_texture_aligned_trajectories(
        trial_bins, LOWER_THRESHOLD, UPPER_THRESHOLD, TIME_WINDOW,
        flip_sides=True, save_fig=save_figures
    )
else:
    side_bias_summary = plot_side_bias_control(
        trial_bins, df_choice, LOWER_THRESHOLD, UPPER_THRESHOLD, TIME_WINDOW,
        save_fig=save_figures
    )

## Export analysis tables

The trial-level table retains the complete label, side assignment, bait texture, mean heading, decision side, and selected texture.

In [ ]:
trial_choice_file = save_path / 'trial_texture_choices.csv'
pairwise_pi_file = save_path / 'pairwise_preference_indices.csv'
df_choice.to_csv(trial_choice_file, index=False)
pairwise_pi.to_csv(pairwise_pi_file, index=False)
print(f'Saved: {trial_choice_file}')
print(f'Saved: {pairwise_pi_file}')